# 04 -- Prediction Models (M1-M5)

**Purpose:** Inspect model performance, feature importance, and test-set
predictions for all five prediction modules.

**Prerequisites:** Run `src/models/run_prediction_modules.py` first.

**Outputs read:** `reports/models/*.csv`

**Module summary:**

| ID | Model | Target | Metric |
|---|---|---|---|
| M1 | LinearRegression | Total student enrolment | R2=0.7784 |
| M2 | Ridge | pillar2_infrastructure (SUBSTITUTE_PROXY) | R2=0.5204 |
| M3 | Holt-Winters per district | enrollment_reported_student_total | R2=0.9967 |
| M4 | RandomForestClassifier | PTR > 30 (teacher shortage) | Acc=0.972 |
| M5 | RandomForestClassifier | risk_tier_v2 (Classroom Infra Proxy) | Acc=0.694 |

Train: 2018-19 to 2023-24 (216 rows). Test: 2024-25 (36 rows).

In [ ]:
from pathlib import Path
import pandas as pd

ROOT    = Path.cwd().parent
RPT_DIR = ROOT / 'reports' / 'models'

summary = pd.read_csv(RPT_DIR / 'all_modules_metrics_summary.csv')
print('Module metrics summary:')
# Show all columns that exist
show_cols = [c for c in ['module', 'model', 'target', 'mae', 'rmse', 'r2', 'accuracy', 'f1_weighted']
             if c in summary.columns]
print(summary[show_cols].to_string(index=False))

In [ ]:
# M1 -- Enrolment / Decline Trend (LinearRegression)
m1_fore = pd.read_csv(RPT_DIR / 'm1_enrolment_forecast.csv')
m1_test = pd.read_csv(RPT_DIR / 'm1_test_predictions.csv')
print(f'M1 forecast rows: {len(m1_fore)}  (expected 72 = 36 districts x 2 forecast years)')
print(m1_fore.head(6).to_string(index=False))

In [ ]:
# M2 -- Learning Outcome SUBSTITUTE PROXY (Ridge on classroom quality)
m2_pred   = pd.read_csv(RPT_DIR / 'm2_learning_proxy_predictions.csv')
m2_status = pd.read_csv(RPT_DIR / 'm2_learning_status.csv')
print('M2 status:')
print(m2_status.to_string(index=False))
print('\nM2 proxy predictions (sample):')
print(m2_pred.head(5).to_string(index=False))

In [ ]:
# M3 -- Total Student Enrolment Forecast (Holt-Winters per district)
# Target: enrollment_reported_student_total  (NOT girls-only enrolment)
m3_fore = pd.read_csv(RPT_DIR / 'm3_total_enrolment_forecast.csv')
m3_test = pd.read_csv(RPT_DIR / 'm3_total_enrolment_test.csv')
print(f'M3 total enrolment forecast rows: {len(m3_fore)}  (expected 72 = 36 districts x 2 years)')
print(m3_fore.head(6).to_string(index=False))
print(f'\nM3 test predictions (2024-25) -- sample:')
print(m3_test.head(5).to_string(index=False))

In [ ]:
# M4 -- Teacher Shortage Risk (RandomForest binary classifier)
# Target: PTR > 30 (RTE norm). PTR is NOT in the feature set.
m4_pred = pd.read_csv(RPT_DIR / 'm4_teacher_shortage_predictions.csv')
m4_fi   = pd.read_csv(RPT_DIR / 'm4_feature_importance.csv')
print('M4 Teacher Shortage Risk -- feature importance:')
print(m4_fi.to_string(index=False))
print('\nM4 test predictions (sorted by risk probability, highest first):')
print(
    m4_pred[['district', 'ptr_students_per_teacher_raw',
              'teacher_shortage_risk', 'predicted_shortage_risk', 'predicted_proba_risk']]
    .sort_values('predicted_proba_risk', ascending=False)
    .head(10).to_string(index=False)
)

In [ ]:
# M5 -- Classroom Infrastructure Quality Trend (Documented Proxy)
# Target: risk_tier_v2.  shala_score_v2 excluded from features (leakage guard).
# NOT a learning-outcome or dropout predictor.
m5_pred = pd.read_csv(RPT_DIR / 'm5_risk_class_predictions.csv')
m5_fi   = pd.read_csv(RPT_DIR / 'm5_feature_importance.csv')
print('M5 Classroom Infra Quality Trend -- feature importance:')
print(m5_fi.to_string(index=False))
print('\nM5 test predictions (sample):')
print(m5_pred[['district', 'shala_score_v2', 'risk_tier_v2', 'predicted_risk_tier']]
      .head(10).to_string(index=False))

In [ ]:
# Confusion matrix for M5
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

tiers = ['At-Risk', 'Critical', 'Stable', 'Thriving']
cm = confusion_matrix(m5_pred['risk_tier_v2'], m5_pred['predicted_risk_tier'], labels=tiers)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(tiers))); ax.set_yticks(range(len(tiers)))
ax.set_xticklabels(tiers, rotation=45, ha='right')
ax.set_yticklabels(tiers)
for i in range(len(tiers)):
    for j in range(len(tiers)):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('M5 Confusion Matrix (2024-25 test set)')
plt.tight_layout()
plt.show()